# Tutorial 01 — Fiber Reorientation

This notebook connects axial orientation, a first-order adaptation law, analytical verification, and the non-unique 90-degree case. All data are synthetic.

## Learning goals

- represent an undirected fiber axis;
- compare Euler integration with the analytical solution;
- quantify adaptation time;
- diagnose the orthogonal-case ambiguity.

## Before running the calculations

This notebook is an interactive learning document. The local package `biomechanics_tutorials` is stored in the repository folder `src/`; it is not an external Anaconda package. The setup cell below finds the repository, adds its source directory to the current kernel when needed, and prints the exact Python interpreter and package file in use.

The plots displayed in this notebook are calculated when the cells run. The PNG and GIF files committed to `figures/` and `animations/` are generated separately by scripts in `experiments/` so that all published outputs can be reproduced without editing the notebook.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates: list[Path] = []

    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])

    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())

    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])

    home = Path.home()
    common_directories = [home, home / "Desktop", home / "Documents", home / "Downloads"]
    for directory in common_directories:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))

    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate

    raise RuntimeError(
        "Repository root was not found. Extract the complete repository, start Jupyter "
        "from its root, or install it in the active environment with: "
        "python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import numpy as np
import matplotlib.pyplot as plt
import biomechanics_tutorials
from biomechanics_tutorials.fiber_reorientation import (
    ReorientationParameters,
    adaptation_time,
    analytical_constant_target,
    angular_difference,
    is_orthogonal,
    simulate_reorientation,
)
from biomechanics_tutorials.localization import tr
from biomechanics_tutorials.plotting import apply_tutorial_style


LANGUAGE = "en"
apply_tutorial_style()
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Python kernel: {sys.executable}")
print(f"Local package: {Path(biomechanics_tutorials.__file__).resolve()}")

## 1. Axial equivalence

A fiber is an axis, so 10° and 190° represent the same orientation.

In [ ]:
np.rad2deg(angular_difference(np.deg2rad(10), np.deg2rad(190)))

## 2. Baseline and analytical verification

In [ ]:
initial = np.deg2rad(-50.0)
target = np.deg2rad(30.0)
params = ReorientationParameters(rate=0.35, duration=20.0, time_step=0.02)
time, numerical, target_history = simulate_reorientation(initial, target, params)
analytical = analytical_constant_target(time, initial, target, params.rate)
discrepancy_deg = np.rad2deg(np.abs(angular_difference(analytical, numerical)))
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(time, np.rad2deg(numerical), label=tr("explicit_euler", LANGUAGE))
axes[0].plot(time, np.rad2deg(analytical), "--", label=tr("analytical", LANGUAGE))
axes[0].legend()
axes[0].set_ylabel(tr("orientation_degrees", LANGUAGE))
axes[1].plot(time, discrepancy_deg)
axes[1].set_ylabel(tr("numerical_discrepancy_degrees", LANGUAGE))
axes[1].set_xlabel(tr("time", LANGUAGE))
plt.show()

In [ ]:
adaptation_time(initial, target, params.rate, np.deg2rad(5.0))

## 3. Orthogonal-case diagnostic

The initial axis -60° is exactly orthogonal to the target 30°. The minimal model cannot select a unique rotation branch.

In [ ]:
is_orthogonal(np.deg2rad(30.0), np.deg2rad(-60.0))

## Reflection

Write one sentence about verification and one sentence about validation before extending the model.

## Rebuild the committed figures and animation

The notebook is intended for interactive exploration and normally does not overwrite repository files. Set the flag below to `True` to run every official experiment script and rebuild the English and Russian outputs.

In [ ]:
REGENERATE_COMMITTED_RESULTS = False

if REGENERATE_COMMITTED_RESULTS:
    import subprocess

    subprocess.run(
        [sys.executable, str(REPOSITORY_ROOT / "tutorials/01-fiber-reorientation/reproduce.py")],
        check=True,
        cwd=REPOSITORY_ROOT,
    )
else:
    message = (
        "Set REGENERATE_COMMITTED_RESULTS = True to rebuild all saved outputs."
        if LANGUAGE == "en"
        else "Установите REGENERATE_COMMITTED_RESULTS = True, чтобы пересоздать все сохранённые результаты."
    )
    print(message)